# Recreate all manuscript and SI figures

This notebook recreates the **four main-text figures** and **eight SI figures** for:

> *Folk melodies carry a mark of vocal performance across historical corpora*

It uses only the committed aggregate outputs and feature tables. It does **not** refit any model or alter any analysis.

## Expected input files

The notebook searches automatically for a folder containing the project output files, including:

- `02_dutch_features.csv`
- `03_dutch_model_results.csv`
- `04_pairwise_results.csv`
- `04_pairwise_coefficients.csv`
- `04_window_model_performance.csv`
- `04_window_coefficients.csv`
- `04_compact_vs_full.csv`
- `05_final_model_comparison.csv`
- `06_finnish_collection_scores.csv`
- `06_primary_core_validation.csv`
- `06_continuity_validation.csv`
- `06_augmented_validation.csv`
- `07_source_type_results.csv`
- `09_symbtr_results.json`
- `09_symbtr_piece_scores.csv`

Figures are saved as both **PNG (300 dpi)** and **vector PDF** under:

- `paper_figures/main/`
- `paper_figures/SI/`


In [ ]:
from pathlib import Path
import ast
import json
import math
import os
import runpy
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from matplotlib.lines import Line2D

warnings.filterwarnings("ignore", category=UserWarning)

# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------

REQUIRED_FILE = "04_pairwise_results.csv"

def _search_roots():
    """The working directory and every directory above it."""
    here = Path.cwd().resolve()
    yield here
    for parent in here.parents:
        yield parent

def find_data_dir():
    """Locate the committed result tables.

    Checks $RESULTS_DIR first, then walks up from the working directory looking
    for results/. Walking up is what lets the notebook run from repo root or
    from notebooks/ without configuration.
    """
    env = os.environ.get("RESULTS_DIR")
    if env and (Path(env) / REQUIRED_FILE).exists():
        return Path(env).resolve()
    for root in _search_roots():
        for candidate in (root / "results", root, root / "03_outputs",
                          root / "project_unzip"):
            if (candidate / REQUIRED_FILE).exists():
                return candidate.resolve()
    raise FileNotFoundError(
        f"Could not locate {REQUIRED_FILE}. Run from inside the repository, or "
        "set RESULTS_DIR to the folder holding the committed result tables."
    )

def find_src_figures():
    """Locate src/figures/, which renders Fig. 4, S7 and S8."""
    for root in _search_roots():
        candidate = root / "src" / "figures"
        if (candidate / "fig4_crosscorpus.py").exists():
            return candidate.resolve()
    return None

DATA_DIR = find_data_dir()
SRC_FIGURES = find_src_figures()
if SRC_FIGURES is not None and str(SRC_FIGURES) not in sys.path:
    sys.path.insert(0, str(SRC_FIGURES))   # so the scripts can import _common
OUTPUT_DIR = Path.cwd() / "paper_figures"
MAIN_DIR = OUTPUT_DIR / "main"
SI_DIR = OUTPUT_DIR / "SI"
MAIN_DIR.mkdir(parents=True, exist_ok=True)
SI_DIR.mkdir(parents=True, exist_ok=True)

print(f"Reading data from: {DATA_DIR}")
print(f"Saving figures to: {OUTPUT_DIR.resolve()}")
print(f"Standalone figure scripts: {SRC_FIGURES}")

# Use a clean, journal-compatible Matplotlib setup.
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

def save_figure(fig, folder, stem):
    """Save a figure as high-resolution PNG and vector PDF."""
    png_path = folder / f"{stem}.png"
    pdf_path = folder / f"{stem}.pdf"
    fig.savefig(png_path, dpi=600, bbox_inches="tight", pad_inches=0.2, facecolor="white")
    fig.savefig(pdf_path, bbox_inches="tight", pad_inches=0.2, facecolor="white")
    print(f"Saved: {png_path}")
    print(f"Saved: {pdf_path}")


## Load committed outputs

In [ ]:
# The figure scripts load whatever tables they need from DATA_DIR. This cell
# only reports what is available, so a missing input is obvious before the
# figures run.
_expected = ["03_dutch_model_results.csv", "04_pairwise_results.csv",
             "04_pairwise_coefficients.csv", "04_window_model_performance.csv",
             "04_window_coefficients.csv", "04_compact_vs_full.csv",
             "05_final_model_comparison.csv", "06_finnish_collection_scores.csv",
             "06_primary_core_validation.csv", "06_continuity_validation.csv",
             "06_augmented_validation.csv", "07_source_type_results.csv",
             "09_symbtr_results.json", "09_symbtr_piece_scores.csv"]
_missing = [f for f in _expected if not (DATA_DIR / f).exists()]
print("Missing required tables:", _missing if _missing else "none")
if not (DATA_DIR / "02_dutch_features.csv").exists():
    print("02_dutch_features.csv absent (not committed): Fig. S2 will be skipped.")


## Main Figure 1 — Study design

In [ ]:
# Fig. 1 is rendered by src/figures/fig1_study_design.py. The notebook calls the
# scripts rather than duplicating them, so there is exactly one implementation
# of each figure and the notebook cannot drift away from the submitted version.
if SRC_FIGURES is None:
    print("Fig. 1 skipped: src/figures/ not found. Run from inside the repository.")
else:
    os.environ["RESULTS_DIR"] = str(DATA_DIR)
    os.environ["FIGURE_DIR"] = str(MAIN_DIR)
    _saved_rc = plt.rcParams.copy()
    try:
        runpy.run_path(str(SRC_FIGURES / "fig1_study_design.py"), run_name="__main__")
    finally:
        plt.rcParams.update(_saved_rc)


## Main Figure 2 — Within-family prediction and era-adjusted coefficients

In [ ]:
# Fig. 2 is rendered by src/figures/fig2_within_family.py. The notebook calls the
# scripts rather than duplicating them, so there is exactly one implementation
# of each figure and the notebook cannot drift away from the submitted version.
if SRC_FIGURES is None:
    print("Fig. 2 skipped: src/figures/ not found. Run from inside the repository.")
else:
    os.environ["RESULTS_DIR"] = str(DATA_DIR)
    os.environ["FIGURE_DIR"] = str(MAIN_DIR)
    _saved_rc = plt.rcParams.copy()
    try:
        runpy.run_path(str(SRC_FIGURES / "fig2_within_family.py"), run_name="__main__")
    finally:
        plt.rcParams.update(_saved_rc)


## Main Figure 3 — Era-window sensitivity and feature stability

In [ ]:
# Fig. 3 is rendered by src/figures/fig3_era_windows.py. The notebook calls the
# scripts rather than duplicating them, so there is exactly one implementation
# of each figure and the notebook cannot drift away from the submitted version.
if SRC_FIGURES is None:
    print("Fig. 3 skipped: src/figures/ not found. Run from inside the repository.")
else:
    os.environ["RESULTS_DIR"] = str(DATA_DIR)
    os.environ["FIGURE_DIR"] = str(MAIN_DIR)
    _saved_rc = plt.rcParams.copy()
    try:
        runpy.run_path(str(SRC_FIGURES / "fig3_era_windows.py"), run_name="__main__")
    finally:
        plt.rcParams.update(_saved_rc)


## Main Figure 4 — Cross-corpus transfer to two target corpora

In [ ]:
# Fig. 4 is rendered by src/figures/fig4_crosscorpus.py. The notebook calls the
# scripts rather than duplicating them, so there is exactly one implementation
# of each figure and the notebook cannot drift away from the submitted version.
if SRC_FIGURES is None:
    print("Fig. 4 skipped: src/figures/ not found. Run from inside the repository.")
else:
    os.environ["RESULTS_DIR"] = str(DATA_DIR)
    os.environ["FIGURE_DIR"] = str(MAIN_DIR)
    _saved_rc = plt.rcParams.copy()
    try:
        runpy.run_path(str(SRC_FIGURES / "fig4_crosscorpus.py"), run_name="__main__")
    finally:
        plt.rcParams.update(_saved_rc)


## SI Figure S1 — Melody-level Dutch classification

In [ ]:
# Fig. S1 is rendered by src/figures/fig_si01_melody_level.py. The notebook calls the
# scripts rather than duplicating them, so there is exactly one implementation
# of each figure and the notebook cannot drift away from the submitted version.
if SRC_FIGURES is None:
    print("Fig. S1 skipped: src/figures/ not found. Run from inside the repository.")
else:
    os.environ["RESULTS_DIR"] = str(DATA_DIR)
    os.environ["FIGURE_DIR"] = str(SI_DIR)
    _saved_rc = plt.rcParams.copy()
    try:
        runpy.run_path(str(SRC_FIGURES / "fig_si01_melody_level.py"), run_name="__main__")
    finally:
        plt.rcParams.update(_saved_rc)


## SI Figure S2 — Feature correlations

In [ ]:
# Fig. S2 is rendered by src/figures/fig_si02_feature_corr.py. The notebook calls the
# scripts rather than duplicating them, so there is exactly one implementation
# of each figure and the notebook cannot drift away from the submitted version.
if SRC_FIGURES is None:
    print("Fig. S2 skipped: src/figures/ not found. Run from inside the repository.")
else:
    os.environ["RESULTS_DIR"] = str(DATA_DIR)
    os.environ["FIGURE_DIR"] = str(SI_DIR)
    _saved_rc = plt.rcParams.copy()
    try:
        runpy.run_path(str(SRC_FIGURES / "fig_si02_feature_corr.py"), run_name="__main__")
    finally:
        plt.rcParams.update(_saved_rc)


## SI Figure S3 — All pairwise models across era windows

In [ ]:
# Fig. S3 is rendered by src/figures/fig_si03_all_models.py. The notebook calls the
# scripts rather than duplicating them, so there is exactly one implementation
# of each figure and the notebook cannot drift away from the submitted version.
if SRC_FIGURES is None:
    print("Fig. S3 skipped: src/figures/ not found. Run from inside the repository.")
else:
    os.environ["RESULTS_DIR"] = str(DATA_DIR)
    os.environ["FIGURE_DIR"] = str(SI_DIR)
    _saved_rc = plt.rcParams.copy()
    try:
        runpy.run_path(str(SRC_FIGURES / "fig_si03_all_models.py"), run_name="__main__")
    finally:
        plt.rcParams.update(_saved_rc)


## SI Figure S4 — Compact versus full model

In [ ]:
# Fig. S4 is rendered by src/figures/fig_si04_compact_vs_full.py. The notebook calls the
# scripts rather than duplicating them, so there is exactly one implementation
# of each figure and the notebook cannot drift away from the submitted version.
if SRC_FIGURES is None:
    print("Fig. S4 skipped: src/figures/ not found. Run from inside the repository.")
else:
    os.environ["RESULTS_DIR"] = str(DATA_DIR)
    os.environ["FIGURE_DIR"] = str(SI_DIR)
    _saved_rc = plt.rcParams.copy()
    try:
        runpy.run_path(str(SRC_FIGURES / "fig_si04_compact_vs_full.py"), run_name="__main__")
    finally:
        plt.rcParams.update(_saved_rc)


## SI Figure S5 — Standardization sensitivity

In [ ]:
# Fig. S5 is rendered by src/figures/fig_si05_standardization.py. The notebook calls the
# scripts rather than duplicating them, so there is exactly one implementation
# of each figure and the notebook cannot drift away from the submitted version.
if SRC_FIGURES is None:
    print("Fig. S5 skipped: src/figures/ not found. Run from inside the repository.")
else:
    os.environ["RESULTS_DIR"] = str(DATA_DIR)
    os.environ["FIGURE_DIR"] = str(SI_DIR)
    _saved_rc = plt.rcParams.copy()
    try:
        runpy.run_path(str(SRC_FIGURES / "fig_si05_standardization.py"), run_name="__main__")
    finally:
        plt.rcParams.update(_saved_rc)


## SI Figure S6 — Leave-one-collection-out cross-corpus transfer

In [ ]:
# Fig. S6 is rendered by src/figures/fig_si06_leave_one_out.py. The notebook calls the
# scripts rather than duplicating them, so there is exactly one implementation
# of each figure and the notebook cannot drift away from the submitted version.
if SRC_FIGURES is None:
    print("Fig. S6 skipped: src/figures/ not found. Run from inside the repository.")
else:
    os.environ["RESULTS_DIR"] = str(DATA_DIR)
    os.environ["FIGURE_DIR"] = str(SI_DIR)
    _saved_rc = plt.rcParams.copy()
    try:
        runpy.run_path(str(SRC_FIGURES / "fig_si06_leave_one_out.py"), run_name="__main__")
    finally:
        plt.rcParams.update(_saved_rc)


## SI Figure S7 — The pitch-core and gap-rate scores make complementary errors

In [ ]:
# Fig. S7 is rendered by src/figures/fig_si_finnish.py. The notebook calls the
# scripts rather than duplicating them, so there is exactly one implementation
# of each figure and the notebook cannot drift away from the submitted version.
if SRC_FIGURES is None:
    print("Fig. S7 skipped: src/figures/ not found. Run from inside the repository.")
else:
    os.environ["RESULTS_DIR"] = str(DATA_DIR)
    os.environ["FIGURE_DIR"] = str(SI_DIR)
    _saved_rc = plt.rcParams.copy()
    try:
        runpy.run_path(str(SRC_FIGURES / "fig_si_finnish.py"), run_name="__main__")
    finally:
        plt.rcParams.update(_saved_rc)


## SI Figure S8 — SymbTr gap rate by form and the aggregation effect

In [ ]:
# Fig. S8 is rendered by src/figures/fig_si_gaprate.py. The notebook calls the
# scripts rather than duplicating them, so there is exactly one implementation
# of each figure and the notebook cannot drift away from the submitted version.
if SRC_FIGURES is None:
    print("Fig. S8 skipped: src/figures/ not found. Run from inside the repository.")
else:
    os.environ["RESULTS_DIR"] = str(DATA_DIR)
    os.environ["FIGURE_DIR"] = str(SI_DIR)
    _saved_rc = plt.rcParams.copy()
    try:
        runpy.run_path(str(SRC_FIGURES / "fig_si_gaprate.py"), run_name="__main__")
    finally:
        plt.rcParams.update(_saved_rc)


## Output inventory

In [ ]:
generated = sorted(OUTPUT_DIR.rglob("*.png")) + sorted(OUTPUT_DIR.rglob("*.pdf"))
print(f"Generated {len(generated)} files:")
for path in generated:
    print(path.relative_to(Path.cwd()))
